In [1]:
import openmc
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
import math

In [ ]:
sys.path.append(os.getcwd())

from materials import get_alfred_materials
from assemblies.fuel_inner import get_fuel_inner_assembly
from assemblies.fuel_outer import get_fuel_outer_assembly
from assemblies.control_rods import get_control_rods_assembly
from assemblies.safety_rods import get_safety_rods_assembly
from assemblies.dummy_elements import get_dummy_assembly

In [ ]:
#materials
openmc.config["cross_sections"] = "/mnt/c/Users/giuli/corso_openmc/ALFRED/endfb71/endfb-vii.1-hdf5/cross_sections.xml" 
mats=get_alfred_materials()

helium = mats['helium']
lead = mats['lead']
ti15_15 = mats['ti15_15']
b4c = mats['b4c']
t91 = mats['t91']
zircaloy = mats['zircaloy']
mox_inner = mats['mox_inner']
mox_outer = mats['mox_outer']

In [ ]:
mats = get_alfred_materials()
materials_file = openmc.Materials(mats.values())
materials_file.export_to_xml()

res_f1 = get_fuel_inner_assembly(mats)
u_f1 = res_f1['universe']

# FUEL OUTER
res_f2 = get_fuel_outer_assembly(mats)
u_f2 = res_f2['universe']

# CONTROL RODS
res_c = get_control_rods_assembly(mats)
u_c = res_c['universe']

# SAFETY RODS
res_s = get_safety_rods_assembly(mats)
u_s = res_s['universe']

# DUMMY ELEMENTS
res_d = get_dummy_assembly(mats)
u_d = res_d['universe']

/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=4.
  warn(msg, IDWarning)
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=5.
  warn(msg, IDWarning)
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance alrea

LATTICE

In [ ]:
lat = openmc.HexLattice()
lat.center = (0., 0.)
lat.pitch = [17.1]
lat.orientation = 'y' 

c_ex = openmc.model.HexagonalPrism(
    edge_length=9.873, 
    orientation='x',
)
c_region=-c_ex
c_cell=openmc.Cell(region=c_region,fill=lead)
c_uni=openmc.Universe(cells=(c_cell,))


lat.universes = [
    [c_uni]*3+[u_d]*5+[c_uni]*5+[u_d]*5+[c_uni]*5+[u_d]*5+[c_uni]*5+[u_d]*5+[c_uni]*5+[u_d]*5+[c_uni]*5+[u_d]*5+[c_uni]*2,
    [c_uni]+[u_d]*8+[c_uni]+[u_d]*8+[c_uni]+[u_d]*8+[c_uni]+[u_d]*8+[c_uni]+[u_d]*8+[c_uni]+[u_d]*8,
    [u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d,u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d,u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d,u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d,u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d,u_d,u_d,u_d,u_f2,u_f2,u_f2,u_d,u_d],
    [u_f2]*42,
    [u_f2,u_f2,u_c,u_f2,u_c,u_f2,u_f2,u_f2,u_c,u_f2,u_c,u_f2,u_f2,u_f2,u_c,u_f2,u_c,u_f2,u_f2,u_f2,u_c,u_f2,u_c,u_f2,u_f2,u_f2,u_c,u_f2,u_c,u_f2,u_f2,u_f2,u_c,u_f2,u_c,u_f2],
    [u_f2]*30,
    [u_f1]*24,
    [u_f1]*18,
    [u_s,u_f1,u_f1,u_s,u_f1,u_f1,u_s,u_f1,u_f1,u_s,u_f1,u_f1],                  
    [u_f1]*6, 
    [u_f1]                         
]
lat.outer = openmc.Universe(cells=[openmc.Cell(fill=lead)])

piano_a=openmc.ZPlane(z0=162, boundary_type='vacuum')
piano_b=openmc.ZPlane(z0=-291.24, boundary_type='vacuum')


r_core=165
r_ext=170
container=openmc.ZCylinder(r = r_core)
cyl_int=openmc.ZCylinder(r = r_core)
cyl_ext=openmc.ZCylinder(r = r_ext, boundary_type='vacuum')
vessel_region=-cyl_ext & +cyl_int & +piano_b & -piano_a
vessel_cell=openmc.Cell(region=vessel_region,fill=t91)
main_cell = openmc.Cell(fill=lat, region=-container & -piano_a & +piano_b)


root_universe = openmc.Universe(cells=[main_cell,vessel_cell])

geometry = openmc.Geometry(root_universe)
geometry.export_to_xml()

PLOT

In [ ]:
#color map
alfred_colors = {
    mox_inner: "yellow", 
    mox_outer: "yellow", 
    helium: "green", 
    lead: "cyan", 
    ti15_15: "blue",  
    t91: "darkorange", 
    zircaloy: "lightgray", 
    b4c: "purple"
}

#plot XY
plot_assembly = openmc.Plot()
plot_assembly.filename = "plot_xy_core"
plot_assembly.basis = 'xy'
plot_assembly.origin = (0,0,71)
plot_assembly.width = (r_core*2, r_core*2)
plot_assembly.pixels = (2000, 2000)
plot_assembly.color_by = 'material'
plot_assembly.colors = alfred_colors

#plot XZ
plot_xz_offset_assembly = openmc.Plot()
plot_xz_offset_assembly.filename = "plot_xz_core"
plot_xz_offset_assembly.basis = 'xz'
plot_xz_offset_assembly.origin = (0,0,-64.62)
plot_xz_offset_assembly.width = (r_core*2, 453.24)
xz_ratio = plot_assembly.width[0] / plot_assembly.width[1]
plot_xz_offset_assembly.pixels = (r_core*2*10,4533)
plot_xz_offset_assembly.color_by = 'material'
plot_xz_offset_assembly.colors = alfred_colors

plots = openmc.Plots([plot_assembly,plot_xz_offset_assembly])
plots.export_to_xml()

openmc.plot_geometry()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

SETTINGS

In [ ]:
r_dist = openmc.stats.Uniform(0.0, r_ext)
phi_dist = openmc.stats.Uniform(0.0, 2*np.pi)
z_dist = openmc.stats.Uniform(-30, 30)

source_dist = openmc.stats.CylindricalIndependent(
    r_dist, 
    phi_dist, 
    z_dist, 
    origin=(0.0, 0.0, 92.5)
)
source = openmc.Source(space=source_dist)
source.only_fissionable = True

settings = openmc.Settings()
settings.source = openmc.Source(space=source_dist)
settings.batches = 1000
settings.inactive = 100
settings.particles = 40000

settings.seed = 1

lower_left,upper_right=geometry.bounding_box
vol_calc=openmc.VolumeCalculation(domains=[mox_inner,mox_outer],samples=10000000,lower_left=lower_left,upper_right=upper_right)
settings.volume_calculations=vol_calc

settings.export_to_xml()

/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/source.py:668: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn(


TALLIES


In [ ]:
import openmc
z_min = 62.5
z_max = 122.5
num_z_bins = 100

mesh_inner = openmc.RegularMesh()
mesh_inner.dimension = [1, 1, num_z_bins] 
mesh_inner.lower_left = [-8.55, -8.55, z_min] 
mesh_inner.upper_right = [8.55, 8.55, z_max]

mesh_inner_filter = openmc.MeshFilter(mesh_inner)

#MESH OUTER ASSEMBLY
mesh_outer = openmc.RegularMesh()
mesh_outer.dimension = [1, 1, num_z_bins] 
mesh_outer.lower_left = [-8.55, 68.4 - 8.55, z_min]   # Y_min = 59.85
mesh_outer.upper_right = [8.55, 68.4 + 8.55, z_max]  # Y_max = 76.95

mesh_outer_filter = openmc.MeshFilter(mesh_outer)

# Tally INNER FLUX
tally_inner_flux = openmc.Tally(name="flux_inner_mesh_alfred")
tally_inner_flux.filters = [mesh_inner_filter]
tally_inner_flux.scores = ['flux']

# Tally OUTER FLUX
tally_outer_flux = openmc.Tally(name="flux_outer_mesh_alfred")
tally_outer_flux.filters = [mesh_outer_filter]  
tally_outer_flux.scores = ['flux']

tallies = openmc.Tallies([tally_inner_flux, tally_outer_flux])
tallies.export_to_xml()

RUN

In [9]:
openmc.run(threads=10)


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

k_eff

In [ ]:
import glob
import os

latest_sp = max(glob.glob('statepoint.*.h5'), key=os.path.getctime)
with openmc.StatePoint(latest_sp) as sp:
    keff_value = sp.keff.n
    keff_std_dev = sp.keff.s
    print(f"k-eff estratto: {keff_value:.5f} +/- {keff_std_dev:.5f}")


keff_data = {
    'keff_mean': [keff_value],
    'keff_std_dev': [keff_std_dev]
}

keff_900K_doppler_1200K = pd.DataFrame(keff_data)
keff_900K_doppler_1200K.to_csv("keff_900K_doppler_1200K.csv", index=False)

print("Dati di keff salvati correttamente.")



/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/uncertainties/core.py:526: FutureWarning: Variable.error_components() is currently an instance method. This method is deprecated. In a future release it will be replaced with an instance property by the same name. It will be accessed by Variable.error_components (with no parentheses).
  return float(sqrt(sum(delta**2 for delta in self.error_components().values())))
/home/giuli/miniconda3/envs/openmc-env/lib/python3.13/site-packages/uncertainties/core.py:491: FutureWarning: Variable.derivatives() is deprecated. It will be removed in a future release.
  for variable, derivative in self.derivatives.items():


k-eff estratto: 1.03656 +/- 0.00011
Dati di keff salvati correttamente.


COSTANTE DOPPLER

In [ ]:
import csv
import math             

T_mox_inner = 900  
T_mox_outer = 900
T_mox_inner_perturbata = 1200
T_mox_outer_perturbata = 1200

T_nom = (1/3*T_mox_inner)+(2/3*T_mox_outer)
T_per = (1/3*T_mox_inner_perturbata)+(2/3*T_mox_outer_perturbata)

def leggi_risultati_csv(nome_file):
    with open(nome_file, mode='r') as file:
        reader = csv.reader(file)

        next(reader) 
        
        dati = next(reader)
        
        k_eff = float(dati[0])
        std_dev = float(dati[1]) 
        
        return k_eff, std_dev

k_nom, err_nom = leggi_risultati_csv("keff_900K_base.csv") 
k_pert, err_pert = leggi_risultati_csv("keff_900K_doppler_1200K.csv")

rho_nom = ((k_nom -1)/k_nom)*(10**5)
rho_pert = ((k_pert -1)/k_pert)*(10**5)

err_rho_nom = (err_nom / (k_nom**2)) * (10**5)
err_rho_pert = (err_pert / (k_pert**2)) * (10**5)

err_delta_rho = math.sqrt(err_rho_nom**2 + err_rho_pert**2)
Kd = (rho_pert - rho_nom) / math.log(T_per/T_nom)
err_Kd = err_delta_rho / abs(math.log(T_per/T_nom))

print(f"Costante di Doppler (KD): {Kd:.5f} ± {err_Kd:.5f} pcm")


Costante di Doppler (KD): -529.11320 ± 52.62051 pcm


LEAD EXPANSION COEFFICIENT

In [ ]:
import csv
import math     


T_lead_nom = 900
T_lead_per = 1200

def leggi_risultati_csv(nome_file1):
    with open(nome_file1, mode='r') as file:
        reader = csv.reader(file)

        next(reader) 
        
        dati = next(reader)
        
        k_eff = float(dati[0])
        std_dev = float(dati[1]) 
        
        return k_eff, std_dev

k_nom_lead, err_nom_lead = leggi_risultati_csv("keff_900K_base.csv") 
k_pert_lead, err_pert_lead = leggi_risultati_csv("keff_900K_lead_1200K.csv")

rho_nom_lead = ((k_nom_lead -1)/k_nom_lead)*(10**5)
rho_pert_lead = ((k_pert_lead -1)/k_pert_lead)*(10**5)

err_rho_nom_lead = (err_nom_lead / (k_nom_lead**2)) * (10**5)
err_rho_pert_lead = (err_pert_lead / (k_pert_lead**2)) * (10**5)

alpha = (rho_pert_lead - rho_nom_lead)/(T_lead_per - T_lead_nom)

err_delta_rho = math.sqrt(err_rho_nom_lead**2 + err_rho_pert_lead**2)

delta_T = T_lead_per - T_lead_nom
err_alpha = err_delta_rho / abs(delta_T)


print(f"Lead expansion coefficient: {alpha:.5f} ± {err_alpha:.5f} pcm/K")

Lead expansion coefficient: -0.64021 ± 0.05161 pcm/K


BURNUP

In [ ]:
import openmc 
import openmc.deplete
geometry=openmc.Geometry.from_xml()
settings=openmc.Settings.from_xml()
model=openmc.Model(geometry=geometry,settings=settings)

openmc.calculate_volumes(threads=8)
results=openmc.VolumeCalculation.from_hdf5("volume_1.h5")
model.geometry.add_volume_information(results)

chain_file="endfb71/endfb-vii.1-hdf5/chain_endfb71_sfr.xml"
operator=openmc.deplete.CoupledOperator(model,chain_file)

dt=[24*60*60]*5
power=1e6

cecm=openmc.deplete.CECMIntegrator(operator,dt,power)
cecm.integrate()

PLOT MESH INNER AND OUTER FUEL 

In [ ]:
import openmc
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

z_min = -30.0  
z_max = 30.0   
num_z_bins = 100
r_max = 0.6
extent_map = [-r_max, r_max, z_min, z_max]
z_centers = np.linspace(z_min, z_max, num_z_bins)


P_termica = 300e6 
E_fission = 200.0 
joule_per_mev = 1.6022e-13
nu_media = 2.5 

fission_rate = P_termica / (E_fission * joule_per_mev)
source_rate = fission_rate * nu_media 

delta_z = (z_max - z_min) / num_z_bins
volume_bin = np.pi * (r_max**2) * delta_z


fattore_conversione = source_rate / volume_bin 


latest_sp = max(glob.glob('statepoint.*.h5'), key=os.path.getctime)

with openmc.StatePoint(latest_sp) as sp:
    
    #plot inner
    tally_in = sp.get_tally(name='flux_inner_mesh_alfred')
    flux_mean_in = tally_in.get_values(scores=['flux'], value='mean').reshape((num_z_bins, 1))
    

    flux_fisico_in = flux_mean_in * fattore_conversione
    flux_norm_in = flux_mean_in / np.max(flux_mean_in)

    fig1, ax1 = plt.subplots(figsize=(7, 9)) 
    im1 = ax1.imshow(flux_norm_in, origin='lower', extent=extent_map, cmap='viridis', vmax=1.0, aspect='auto')
    cbar1 = fig1.colorbar(im1, ax=ax1)
    cbar1.set_label('Flusso Normalizzato (max=1)', fontsize=12)
    ax1.set_ylabel('Altezza Assiale Z [cm]', fontsize=12)
    ax1.set_xlabel('Posizione X (Radiale) [cm]', fontsize=12)
    ax1.set_yticks(np.linspace(z_min, z_max, 7))
    ax1.set_title('Mappa del flusso - Inner Pin')
    plt.tight_layout()
    plt.savefig('flusso_alfred_inner_mappa.png') 
    plt.close() 

    fig1b, ax1b = plt.subplots(figsize=(8, 6))
    ax1b.plot(flux_fisico_in.flatten(), z_centers, color='blue', linewidth=2)
    

    ax1b.set_xlabel(r'Flusso [$cm^{-2} \cdot s^{-1}$]', fontsize=12) 
    ax1b.set_ylabel('Altezza Assiale Z [cm]', fontsize=12)
    ax1b.set_title('Distribuzione assiale flusso - Inner Pin')
    plt.tight_layout()
    plt.savefig('flusso_alfred_inner_linea.png')
    plt.close()


    #plot inner
    tally_out = sp.get_tally(name='flux_outer_mesh_alfred')
    flux_mean_out = tally_out.get_values(scores=['flux'], value='mean').reshape((num_z_bins, 1))
    
    flux_fisico_out = flux_mean_out * fattore_conversione

    flux_norm_out = flux_mean_out / np.max(flux_mean_out)

    fig2, ax2 = plt.subplots(figsize=(7, 9)) 
    im2 = ax2.imshow(flux_norm_out, origin='lower', extent=extent_map, cmap='viridis', vmax=1.0, aspect='auto')
    cbar2 = fig2.colorbar(im2, ax=ax2)
    cbar2.set_label('Flusso Normalizzato (max=1)', fontsize=12)
    ax2.set_ylabel('Altezza Assiale Z [cm]', fontsize=12)
    ax2.set_xlabel('Posizione X (Radiale) [cm]', fontsize=12)
    ax2.set_yticks(np.linspace(z_min, z_max, 7))
    ax2.set_title('Mappa del flusso - Outer Pin')
    plt.tight_layout()
    plt.savefig('flusso_alfred_outer_mappa.png') 
    plt.close()

    fig2b, ax2b = plt.subplots(figsize=(8, 6))
    ax2b.plot(flux_fisico_out.flatten(), z_centers, color='red', linewidth=2)
    
    ax2b.set_xlabel(r'Flusso [$cm^{-2} \cdot s^{-1}$]', fontsize=12)
    ax2b.set_ylabel('Altezza Assiale Z [cm]', fontsize=12)
    ax2b.set_title('Distribuzione assiale flusso - Outer Pin')
    plt.tight_layout()
    plt.savefig('flusso_alfred_outer_linea.png')
    plt.close()

print("Elaborazione completata. Grafici salvati con flusso fisico (cm^-2 s^-1).")
    

REACTIVITY GRAPH AS CONTROL RODS HEIGHT VARY

In [ ]:
import csv
import openmc
from scipy.interpolate import CubicSpline
import numpy as np



def leggi_risultati_csv(nome_file1):
    with open(nome_file1, mode='r') as file:
        reader = csv.reader(file)

        next(reader) 
        
        dati = next(reader)
        
        k_eff = float(dati[0])
        
        return k_eff
    
#plot
k_0 = leggi_risultati_csv("k_0.csv") 
k_1 = leggi_risultati_csv("k_1.csv") 
k_2 = leggi_risultati_csv("k_2.csv") 
k_3 = leggi_risultati_csv("k_3.csv") 
k_4 = leggi_risultati_csv("k_4.csv") 
k_5 = leggi_risultati_csv("k_5.csv") 
k_6 = leggi_risultati_csv("k_6.csv") 
k_7 = leggi_risultati_csv("k_7.csv") 
k_8 = leggi_risultati_csv("k_8.csv") 
k_9 = leggi_risultati_csv("k_9.csv") 
k_10 = leggi_risultati_csv("k_10.csv") 

rho = []

rho.append(((k_0 -1)/k_0)*(10**5))
rho.append(((k_1 -1)/k_1)*(10**5))
rho.append(((k_2 -1)/k_2)*(10**5))
rho.append(((k_3 -1)/k_3)*(10**5))
rho.append(((k_4 -1)/k_4)*(10**5))
rho.append(((k_5 -1)/k_5)*(10**5))
rho.append(((k_6 -1)/k_6)*(10**5))
rho.append(((k_7 -1)/k_7)*(10**5))
rho.append(((k_8 -1)/k_8)*(10**5))
rho.append(((k_9 -1)/k_9)*(10**5))
rho.append(((k_10 -1)/k_10)*(10**5))

rho = np.array(rho)

inserimento_cr = np.array([62.5, 62.5+6, 62.5+12, 62.5+18, 62.5+24, 62.5+30, 62.5+36, 62.5+42, 62.5+48, 62.5+54, 62.5+60])

funzione_rho_3 = CubicSpline(inserimento_cr, rho)
x_liscio = np.linspace(min(inserimento_cr), max(inserimento_cr), 1000)
y_liscio = funzione_rho_3(x_liscio)

plt.figure()
plt.plot(x_liscio, y_liscio)
plt.scatter(inserimento_cr, rho, color='red', zorder=5, label="Punti Simulati")
plt.grid(True, linestyle='--', alpha=0.6)
plt.xlabel('Altezza barre di controllo [cm]')
plt.ylabel('Reattività [pcm]')
plt.title('Reattività al variare dell''altezza delle CRs')
plt.tight_layout()
plt.savefig('reattvità al variare dell''altezza delle CRs.png') 
plt.close()


TOTAL WORTH OF 12 CRs

In [ ]:
import csv
import openmc
from scipy.interpolate import CubicSpline
import numpy as np


def leggi_risultati_csv(nome_file1):
    with open(nome_file1, mode='r') as file:
        reader = csv.reader(file)

        next(reader) 
        
        dati = next(reader)
        
        k_eff = float(dati[0])
        
        return k_eff

k_10 = leggi_risultati_csv("k_10.csv") 
k_0 = leggi_risultati_csv("k_0.csv") 
rho_out = ((k_0 - 1)/k_0)* 10**5
rho_in = ((k_10 - 1)/k_10)* 10**5
rho_tot = rho_in - rho_out

print(f"Total worth of 12 CRs: {rho_tot:.5f} pcm")

Total worth of 12 CRs: -10706.52580 pcm


In [3]:
import csv
import openmc
from scipy.interpolate import CubicSpline
import numpy as np
def leggi_risultati_csv(nome_file1):
    with open(nome_file1, mode='r') as file:
        reader = csv.reader(file)

        next(reader) 
        
        dati = next(reader)
        
        k_eff = float(dati[0])
        
        return k_eff
keff_900K_base = leggi_risultati_csv("keff_900K_base.csv") 
rho = ((keff_900K_base - 1)/keff_900K_base)* 10**5
print(f"Reattività: {rho:.5f} pcm")

Reattività: 3678.91647 pcm
